In [4]:
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
import duckdb

# 1. Configurar las credenciales de Google
CONFIG_SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

credenciales = Credentials.from_service_account_file(
    'credenciales.json', 
    scopes=CONFIG_SCOPES
)

# 2. Conectarse a la Base de Datos Cruda (Google Sheets)
cliente_gmail = gspread.authorize(credenciales)
# IMPORTANTE: Asegúrate de que este nombre sea EXACTO al de tu archivo de Google Sheets
sheet_id = cliente_gmail.open('DB_Tienda_Operacional')

# 3. Extraer los datos de la pestaña 'Ventas'
hoja_ventas = sheet_id.worksheet('Ventas')
datos_crudos = hoja_ventas.get_all_records()

# 4. Convertirlos en un DataFrame de Pandas (Estructura de tabla en Python)
df_ventas = pd.DataFrame(datos_crudos)

# 5. Mostrar los primeros registros para validar que funcionó
df_ventas.head()

,id_venta,fecha_hora,metodo_pago
0,3061665f,01/01/2009 7:07:07,QR
1,9fd93aba,23/03/2020 7:08:15,Efectivo


In [5]:
# 1. Extraer el resto de las tablas de Google Sheets
hoja_detalle = sheet_id.worksheet('Detalle_Ventas')
hoja_productos = sheet_id.worksheet('Productos')
hoja_gastos = sheet_id.worksheet('Gastos')

# 2. Convertirlas en DataFrames de Pandas
df_detalle = pd.DataFrame(hoja_detalle.get_all_records())
df_productos = pd.DataFrame(hoja_productos.get_all_records())
df_gastos = pd.DataFrame(hoja_gastos.get_all_records())

print("✅ Todas las tablas extraídas con éxito al Data Lake en memoria.")

# 3. Usar DuckDB para cruzar Ventas, Detalles y Productos con SQL
query_ventas_completas = """
    SELECT 
        v.fecha_hora,
        v.metodo_pago,
        d.id_producto,
        p.nombre,
        d.cantidad,
        p.precio_compra,
        p.precio_venta,
        d.subtotal,
        -- Calculamos la ganancia bruta por producto vendido (Precio Venta - Precio Compra) * Cantidad
        (p.precio_venta - p.precio_compra) * d.cantidad AS ganancia_bruta
    FROM df_ventas AS v
    JOIN df_detalle AS d ON v.id_venta = d.id_venta
    JOIN df_productos AS p ON d.id_producto = p.id_producto
"""

# Ejecutar la consulta SQL y guardar el resultado en un nuevo DataFrame
df_modelo_ventas = duckdb.query(query_ventas_completas).to_df()

# Mostrar el resultado del cruce
df_modelo_ventas.head()

✅ Todas las tablas extraídas con éxito al Data Lake en memoria.


,fecha_hora,metodo_pago,id_producto,nombre,cantidad,precio_compra,precio_venta,subtotal,ganancia_bruta
0,01/01/2009 7:07:07,QR,9902f327,Coca Cola 600ml,2,5,8,16,6
1,23/03/2020 7:08:15,Efectivo,9902f327,Coca Cola 600ml,3,5,8,24,9


In [8]:
# 1. Intentar crear la pestaña 'BI_Ventas_Modeladas', si ya existe, la abrimos y la limpiamos
try:
    hoja_destino = sheet_id.add_worksheet(title="BI_Ventas_Modeladas", rows="1000", cols="20")
    print("Pestaña 'BI_Ventas_Modeladas' creada desde cero.")
except Exception:
    hoja_destino = sheet_id.worksheet("BI_Ventas_Modeladas")
    hoja_destino.clear()
    print("Pestaña existente encontrada. Datos antiguos limpiados con éxito.")

# 2. Preparar los datos convirtiendo el DataFrame a una lista de listas (formato que entiende Google)
columnas = df_modelo_ventas.columns.values.tolist()
filas = df_modelo_ventas.values.tolist()
datos_a_subir = [columnas] + filas

# 3. Subir los datos modelados a la nube
hoja_destino.update(range_name='A1', values=datos_a_subir)

print("ETL COMPLETADO! Los datos analíticos ya están disponibles en Google Sheets para el Dashboard.")

Pestaña existente encontrada. Datos antiguos limpiados con éxito.
ETL COMPLETADO! Los datos analíticos ya están disponibles en Google Sheets para el Dashboard.
